# E4, E5, and E6 follow-up analyses

Colab workflow for repeated group splits (E4), transition matching sensitivity (E5), and probability calibration (E6). Add `GITHUB_TOKEN` and `HF_TOKEN` as Colab secrets. Large artifacts stay on Drive; only an optional Markdown result record is pushed.


In [ ]:
import os, shutil, stat, subprocess, sys
from contextlib import contextmanager
from pathlib import Path
from google.colab import drive, userdata

def secret(name):
    value = userdata.get(name)
    if not value: raise RuntimeError(f"Add Colab secret {name}")
    return value
GITHUB_TOKEN, HF_TOKEN = secret("GITHUB_TOKEN"), secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

@contextmanager
def git_auth():
    path = Path("/content/git-askpass.sh")
    path.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
    path.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
    env = os.environ | {"GITHUB_TOKEN": GITHUB_TOKEN, "GIT_ASKPASS": str(path), "GIT_TERMINAL_PROMPT": "0"}
    try: yield env
    finally: path.unlink(missing_ok=True)

REPO = "https://github.com/sagnikc395/tracing-math.git"
BRANCH = "main"
PROJECT_ROOT = Path("/content/tracing-math")
if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
with git_auth() as env:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, str(PROJECT_ROOT)], check=True, env=env)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)], check=True)
# Colab: make editable install importable without restart
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
try:
    import tracing_math
    print(f"tracing_math OK: {tracing_math.__file__}")
except ModuleNotFoundError as _e:
    raise RuntimeError(f"pip install -e failed — restart runtime and re-run. {_e}") from _e
subprocess.run(["git", "config", "user.name", "sagnikc395"], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "config", "user.email", "sagnikchatterjee607@gmail.com"], cwd=PROJECT_ROOT, check=True)


In [ ]:
import json
import numpy as np
import pandas as pd
import yaml
from datetime import datetime, timezone
from IPython.display import display
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/math-error-tracing")
DATA = DRIVE / "data/processbench.jsonl"
ARTIFACTS = DRIVE / "artifacts/qwen2.5-math-1.5b-a100-bf16"
ANALYSIS = DRIVE / "e4-e5-e6/artifacts/analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load((PROJECT_ROOT / "configs/project.yaml").read_text())
config["data"]["output_path"] = str(DATA)
config["extraction"]["output_dir"] = str(ARTIFACTS)
config["artifacts"]["analysis_dir"] = str(ANALYSIS)
CONFIG = Path("/content/e4_e5_e6.yaml")
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
if not DATA.exists() or not (ARTIFACTS / "activation_shards").exists():
    raise FileNotFoundError("This notebook needs the primary ProcessBench file and activation shards on Drive.")
def cli(*args):
    command = [sys.executable, "-m", "tracing_math", "--config", str(CONFIG), *args]
    print("$", " ".join(command)); subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## E4: split sensitivity

This cell uses five fixed seeds. For each seed it reassigns complete problem groups, selects a hidden layer and C on validation using trace-equal training weights, then refits the five conditional feature sets. It is CPU/GPU-memory intensive because it loads cached activations.


In [ ]:
import shutil
import sys
from pathlib import Path as _P
# Fallback if setup cell was skipped or kernel restarted
_ROOT = _P("/content/tracing-math")
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from tracing_math.conditional import conditional_hidden_state_analysis
from tracing_math.data import assign_partitions, load_traces
from tracing_math.pipeline import load_activation_shards
from tracing_math.probes import change_point_metrics, choose_threshold

# --- SPEED PILOT: set to (7,13,29,43,61) for full 5-seed run (~30-45min) ---
SPLIT_SEEDS = (7,)  # pilot 1 seed ~5-9min; full restores 5 seeds
# Cache Drive shards to local SSD once (3-8min copy, then 10x faster loads)
if str(ARTIFACTS).startswith("/content/drive"):
    LOCAL = _P("/tmp/artifacts")
    if not (LOCAL / "activation_shards").exists():
        print(f"Copying {ARTIFACTS} -> {LOCAL} (one-time, ~2GB)...")
        shutil.copytree(ARTIFACTS, LOCAL, dirs_exist_ok=True)
    ARTIFACTS_LOCAL = LOCAL
else:
    ARTIFACTS_LOCAL = ARTIFACTS
activations, metadata = load_activation_shards(ARTIFACTS_LOCAL)
# Make next cells (E5) read from local copy too
ARTIFACTS = ARTIFACTS_LOCAL
config["extraction"]["output_dir"] = str(ARTIFACTS_LOCAL)
import yaml as _yaml; _P("/content/e4_e5_e6.yaml").write_text(_yaml.safe_dump(config, sort_keys=False))
# Keep traces on Drive (small: 7.6MB) - no need to copy
traces = load_traces(DATA)

def trace_weights(frame):
    counts = frame.trace_id.value_counts()
    return 1 / frame.trace_id.map(counts).to_numpy(float)

def choose_layer(frame, seed):
    masks = {p: frame.partition.eq(p).to_numpy() for p in ("train", "validation")}
    y = frame.invalid_so_far.to_numpy(int)
    rows = []
    for layer in range(activations.shape[1]):
        x = activations[:, layer, :].astype(np.float32)
        scaler = StandardScaler().fit(x[masks["train"]])
        # C) FIX: transform once per layer, not per C (4x speedup)
        Xt = scaler.transform(x[masks["train"]])
        Xv = scaler.transform(x[masks["validation"]])
        y_train, y_val = y[masks["train"]], y[masks["validation"]]
        w_train = trace_weights(frame.loc[masks["train"]])
        best = None
        for c in config["analysis"]["control_c_values"]:
            model = LogisticRegression(C=c, class_weight="balanced", max_iter=config["analysis"]["transition_max_iter"], solver="liblinear", random_state=seed)
            model.fit(Xt, y_train, sample_weight=w_train)
            scores = model.predict_proba(Xv)[:, 1]
            candidate = (roc_auc_score(y_val, scores), -abs(np.log10(c)), c, scores)
            best = max(best, candidate, key=lambda item: item[:2]) if best else candidate
        threshold = choose_threshold(frame.loc[masks["validation"]], best[3])
        process_f1 = change_point_metrics(frame.loc[masks["validation"]], best[3], threshold)["process_f1"]
        rows.append((process_f1, best[0], -layer, layer))
    return max(rows)[3]

metric_frames, interval_frames, selection_rows = [], [], []
for split_seed in SPLIT_SEEDS:
    assignments = assign_partitions(traces, seed=split_seed, train_fraction=config["probe"]["train_fraction"], validation_fraction=config["probe"]["validation_fraction"])
    frame = metadata.copy(); frame["partition"] = frame.trace_id.map(assignments)
    # B) for full 5-seed speed: uncomment to skip 112-fit search and reuse frozen layer 23
    # layer = int(np.load(ARTIFACTS_LOCAL / "probes/directions.npz")["selected_layer"])  # ~0min vs 3.7min/seed
    layer = choose_layer(frame, split_seed)
    result = conditional_hidden_state_analysis(activations, frame, traces, layer=layer, c_values=tuple(config["analysis"]["control_c_values"]), max_iter=config["analysis"]["transition_max_iter"], bootstrap_samples=config["analysis"]["transition_bootstrap_samples"], confidence_level=config["analysis"]["confidence_level"], seed=split_seed, tfidf_min_df=config["analysis"]["conditional_tfidf_min_df"], tfidf_max_features=config["analysis"]["conditional_tfidf_max_features"])
    metrics = result.metrics.copy(); metrics["split_seed"] = split_seed; metrics["selected_layer"] = layer; metric_frames.append(metrics)
    intervals = result.paired_intervals.copy(); intervals["split_seed"] = split_seed; intervals["selected_layer"] = layer; interval_frames.append(intervals)
    selection_rows.append({"split_seed": split_seed, "selected_layer": layer})
E4_ROOT = ANALYSIS / "split_sensitivity"; E4_ROOT.mkdir(exist_ok=True)
pd.concat(metric_frames).to_csv(E4_ROOT / "metrics_by_seed.csv", index=False)
pd.concat(interval_frames).to_csv(E4_ROOT / "paired_intervals_by_seed.csv", index=False)
pd.DataFrame(selection_rows).to_csv(E4_ROOT / "selection_by_seed.csv", index=False)
display(pd.concat(metric_frames).pivot(index="split_seed", columns="condition", values="auroc").round(4))


## E5: transition matching sensitivity


In [ ]:
cli("transition-sensitivity")
E5_ROOT = ANALYSIS / "transition_probe"
display(pd.read_csv(E5_ROOT / "matching_sensitivity.csv").round(4))
display(pd.read_csv(E5_ROOT / "matching_sensitivity_diagnostics.csv").round(4))


## E6: probability calibration

This reports Brier score, log loss, ECE under several bin counts, and a reliability diagram. It does not call threshold transfer calibration.


In [ ]:
import matplotlib.pyplot as plt
from tracing_math.probes import binary_metrics, calibration_curve_table
predictions = pd.read_csv(ARTIFACTS / "probes/test_predictions.csv")
labels, scores = predictions.label.to_numpy(int), predictions.score.to_numpy(float)
rows = []
for bins in (5, 10, 15, 20):
    rows.append({"bins": bins, **binary_metrics(labels, scores, calibration_bins=bins)})
calibration = calibration_curve_table(labels, scores, bins=10)
E6_ROOT = ANALYSIS / "calibration"; E6_ROOT.mkdir(exist_ok=True)
pd.DataFrame(rows).to_csv(E6_ROOT / "metrics_by_bins.csv", index=False)
calibration.to_csv(E6_ROOT / "reliability_bins.csv", index=False)
plt.figure(figsize=(5, 5)); plt.plot([0, 1], [0, 1], "--", color="gray")
nonempty = calibration[calibration.n > 0]
plt.plot(nonempty.mean_score, nonempty.positive_rate, marker="o")
plt.xlabel("Mean predicted probability"); plt.ylabel("Empirical invalid-so-far rate"); plt.title("Reliability diagram")
plt.tight_layout(); plt.savefig(E6_ROOT / "reliability_diagram.png", dpi=200); plt.show()
display(pd.DataFrame(rows)[["bins", "brier_score", "log_loss", "expected_calibration_error"]].round(4))


In [ ]:
PUSH_RESULTS = False
tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
record = PROJECT_ROOT / "results" / f"e4_e5_e6_{tag}.md"
record.write_text(f"# E4, E5, and E6 follow-up\n\nRun: `{tag}`\n\nArtifacts are stored on Drive at `{ANALYSIS}`.\n")
if PUSH_RESULTS:
    subprocess.run(["git", "add", "--", str(record.relative_to(PROJECT_ROOT))], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "commit", "-m", "results: add E4 E5 E6 record"], cwd=PROJECT_ROOT, check=True)
    with git_auth() as env: subprocess.run(["git", "push", "origin", BRANCH], cwd=PROJECT_ROOT, check=True, env=env)
else: print(record)
